In [1]:
from PIL import Image, ImageEnhance
import math
import os

def img_watermark(image_name, image_path):
    # 상위 폴더 경로 계산
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    
    # 출력 폴더 생성 (없을 경우)
    os.makedirs(output_dir, exist_ok=True)
    
    # 원본 이미지 열기
    image = Image.open(image_path).convert("RGBA")
    width, height = image.size

    # 워터마크 로고 열기 (알파 채널 보존)
    logo = Image.open("logo.png").convert("RGBA")
    
    # 로고 투명도 50% 조정 (원본 알파값 * 0.5)
    alpha = logo.split()[3]
    alpha = ImageEnhance.Brightness(alpha).enhance(0.5)
    logo.putalpha(alpha)

    # 워터마크 레이어 생성
    watermark_layer = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    logo_width, logo_height = logo.size

    # 워터마크 간격 설정
    interval_x = math.trunc(width / 35)*10 if width > 600 else 200
    interval_y = 200 if height > 600 else math.trunc(height / 30)*10

    # 워터마크 반복 배치
    for y in range(0, height + interval_y, interval_y):
        for x in range(0, width + interval_x, interval_x):
            watermark_layer.paste(logo, (x, y), logo)

    # 전체 워터마크 레이어 45도 회전
    rotated_watermark_layer = watermark_layer.rotate(45, expand=True, center=(width//2, height//2))

    # 원본과 합성 (투명도 자동 적용)
    watermarked_image = Image.alpha_composite(
        image,
        rotated_watermark_layer.resize(image.size)
    )

    # 저장 경로 생성
    save_path = os.path.join(output_dir, f"wm_{image_name}")

    # 결과 저장
    watermarked_image.save(save_path)

    # 저장 경로 반환
    return save_path


In [2]:
import os
import fitz  # PyMuPDF 임포트
def pdf_watermark(pdf_name, path):
    
    # 상위 폴더 경로 계산
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    
    # 출력 폴더 생성 (없을 경우)
    os.makedirs(output_dir, exist_ok=True)
    
    # 파일 경로 설정
    original_file = path
    watermark_file = 'WaterMark.pdf'
    new_file = os.path.join(output_dir, f"wm_{pdf_name}")
    
    # PDF 워터마킹 처리
    original_pdf = fitz.open(original_file)
    watermark_pdf = fitz.open(watermark_file)
    
    for page_num in range(len(original_pdf)):
        page = original_pdf[page_num]
        page.show_pdf_page(page.rect, watermark_pdf, 0)
    
    original_pdf.save(new_file)
    return new_file  # 전체 저장 경로 반환

In [3]:
# !pip install flask

In [ ]:
from flask import Flask, request
from flask import Response
from flask import jsonify
from concurrent.futures import ThreadPoolExecutor

app = Flask(__name__)
app.config['JSON_AS_ASCII'] = False


@app.route('/watermark', methods=['GET'])
def watermark():
    try:
        files = request.get_json()['files']
        print("========================================================")
        print("받은 files:", files)
        if not files or not isinstance(files, list):
            return jsonify({"error": "Invalid file list"}), 400

        file_info = [
            (file['filename'], file['path'], file['path'].split(".")[-1].lower())
            for file in files
        ]

        print("========================================================")
        print("정제한 files:", file_info)

        # 실제 워터마크 처리 로직 (예시)
        processed_paths = []
        with ThreadPoolExecutor() as executor:
            futures = []
            for filename, path, ext in file_info:
                if ext == 'pdf':
                    futures.append(executor.submit(pdf_watermark, filename, path))
                else:
                    futures.append(executor.submit(img_watermark, filename, path))
            
            for future in futures:
                result = future.result()
                processed_paths.append(result)

        return jsonify({"wm_path": processed_paths}), 200

    except Exception as e:
        print(e)
        return jsonify({"error": str(e)}), 500



# 코드수정시 자동반영
if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
